# Bank Campaign Sense
**Company:** Brainybeam Info-Tech PVT LTD  
**Domain:** Data Science & Machine Learning with Data Analytics  
**Tools:** Jupyter Notebook | Python | Power BI

---

## Project Abstract

Predictive analytics plays a crucial role in modern bank marketing campaigns. By harnessing historical customer data — demographics, transaction history, and previous campaign responses — this project builds a model that identifies customers most likely to subscribe to a term deposit.

The goal is simple: help the bank stop wasting money calling uninterested people, and focus effort on leads that convert.

---
**Keywords:** Python, Pandas, NumPy, Scikit-learn, Matplotlib, Seaborn, Classification, Power BI  
**Language:** Python | **Tools:** Jupyter Notebook

---
## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

print("Libraries loaded.")

---
## Step 2: Load Data

The dataset `bank.xls` uses `.xls` extension but is actually comma-separated — so we load it with `pd.read_csv`. It contains customer demographics, financial details, contact history, and a target column `deposit` indicating if the client subscribed to a term deposit.

In [ ]:
df = pd.read_csv('bank.xls')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
print(df.info())
df.describe()

---
## Step 3: Data Cleaning

In [ ]:
print("Null values:")
null_counts = df.isnull().sum()
print(null_counts)
print(f"\nTotal nulls: {null_counts.sum()}")

In [ ]:
initial_count = len(df)
df = df.drop_duplicates()
print(f"Duplicates removed: {initial_count - len(df)}")
print(f"Shape after cleaning: {df.shape}")

In [ ]:
print("'unknown' counts in categorical columns:")
for col in df.select_dtypes(include='object').columns:
    count = (df[col] == 'unknown').sum()
    if count > 0:
        print(f"  {col}: {count}")

In [ ]:
print(df.dtypes)

---
## Step 4: Export Cleaned Data for Power BI

We export the cleaned dataset so stakeholders can build dashboards in Power BI without needing to touch code.

> **Power BI Steps:**  
> 1. Open Power BI Desktop → Get Data → CSV → `cleaned_bank_data.csv`  
> 2. Create: Pie chart (Target), Bar chart (Job vs Response), Line chart (Age vs Balance)  
> 3. Add slicers for Education, Marital Status, Job Type  
> 4. Publish to Power BI Service

In [ ]:
df.to_csv('cleaned_bank_data.csv', index=False)
print(f"Exported cleaned_bank_data.csv — {len(df)} rows, {df.shape[1]} columns")

---
## Step 5: Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.countplot(x='deposit', data=df, palette='Set2', edgecolor='black')
plt.title('Target Distribution (Campaign Response)', fontsize=13, fontweight='bold')
plt.xlabel('Subscribed to Term Deposit')
plt.ylabel('Count')
for p in plt.gca().patches:
    plt.gca().annotate(f'{int(p.get_height())}',
                       (p.get_x() + p.get_width()/2., p.get_height()),
                       ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.subplot(1, 2, 2)
df['deposit'].value_counts().plot(kind='pie', autopct='%1.1f%%',
                             colors=['#66c2a5', '#fc8d62'],
                             startangle=90, explode=[0, 0.05])
plt.title('Target Class Proportion', fontsize=13, fontweight='bold')
plt.ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.histplot(df['age'], kde=True, color='steelblue', bins=30)
plt.title('Customer Age Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Age')
plt.ylabel('Count')

plt.subplot(1, 2, 2)
sns.histplot(df['balance'].clip(-1000, 5000), kde=True, color='coral', bins=30)
plt.title('Account Balance Distribution (Clipped)', fontsize=13, fontweight='bold')
plt.xlabel('Balance')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
job_response = df.groupby(['job', 'deposit']).size().unstack()
job_response.plot(kind='bar', ax=plt.gca(), color=['#fc8d62', '#66c2a5'], edgecolor='black')
plt.title('Job Type vs Campaign Response', fontsize=13, fontweight='bold')
plt.xlabel('Job Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(['No', 'Yes'], title='Subscribed')

plt.subplot(1, 2, 2)
edu_response = df.groupby(['education', 'deposit']).size().unstack()
edu_response.plot(kind='bar', ax=plt.gca(), color=['#fc8d62', '#66c2a5'], edgecolor='black')
plt.title('Education vs Campaign Response', fontsize=13, fontweight='bold')
plt.xlabel('Education Level')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.legend(['No', 'Yes'], title='Subscribed')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
numeric_df = df.select_dtypes(include=np.number)
corr_matrix = numeric_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap (Numerical Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='deposit', y='duration', data=df, palette='pastel')
plt.title('Call Duration vs Campaign Response', fontsize=13, fontweight='bold')
plt.xlabel('Subscribed to Term Deposit')
plt.ylabel('Last Contact Duration (seconds)')
plt.show()

---
## Step 6: Feature Encoding & Preprocessing

- Target `y` mapped to 0/1  
- Nominal categoricals one-hot encoded  
- All features scaled with `StandardScaler` (important for KNN, Logistic Regression)  
- 80/20 stratified train-test split

In [ ]:
df_encoded = df.copy()
df_encoded['deposit'] = df_encoded['deposit'].map({'yes': 1, 'no': 0})

print("Target encoding:")
print(df_encoded['deposit'].value_counts())

In [ ]:
y = df_encoded['deposit']
X = df_encoded.drop('deposit', axis=1)

cat_cols = X.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {cat_cols}")

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print(f"\nFeatures before encoding: {X.shape[1]}")
print(f"Features after encoding : {X_encoded.shape[1]}")

In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_encoded), columns=X_encoded.columns)

print(f"Scaling done — mean: {X_scaled.iloc[:, 0].mean():.4f}, std: {X_scaled.iloc[:, 0].std():.4f}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
print(f"\nTrain class dist:\n{y_train.value_counts()}")
print(f"\nTest class dist:\n{y_test.value_counts()}")

---
## Step 7: Train Classification Models

Eight algorithms trained and compared:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. K-Nearest Neighbors
5. Naive Bayes
6. AdaBoost
7. Gradient Boosting
8. XGBoost

In [ ]:
models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree"       : DecisionTreeClassifier(random_state=42, max_depth=6),
    "Random Forest"       : RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN"                 : KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes"         : GaussianNB(),
    "AdaBoost"            : AdaBoostClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting"   : GradientBoostingClassifier(n_estimators=100, random_state=42),
    "XGBoost"             : XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)
}

trained_models = {}
print("Training models...\n")
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"  {name} — done")

print("\nAll models trained.")

---
## Step 8: Evaluation & Comparative Analysis

In [ ]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    results.append({
        "Model"     : name,
        "Accuracy"  : round(accuracy_score(y_test, y_pred) * 100, 2),
        "Precision" : round(precision_score(y_test, y_pred, zero_division=0) * 100, 2),
        "Recall"    : round(recall_score(y_test, y_pred, zero_division=0) * 100, 2),
        "F1-Score"  : round(f1_score(y_test, y_pred, zero_division=0) * 100, 2)
    })
    print(f"{'='*55}")
    print(f" {name}")
    print(f"{'='*55}")
    print(classification_report(y_test, y_pred, target_names=['No (0)', 'Yes (1)']))

In [ ]:
comparison_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)
comparison_df.index += 1

print("=" * 65)
print("       COMPARATIVE ANALYSIS — ALL CLASSIFICATION MODELS")
print("=" * 65)
print(comparison_df.to_string())
print("=" * 65)

best_model_name = comparison_df.iloc[0]['Model']
best_f1         = comparison_df.iloc[0]['F1-Score']
best_acc        = comparison_df.iloc[0]['Accuracy']

print(f"\nBest Model : {best_model_name}")
print(f"F1-Score   : {best_f1}%")
print(f"Accuracy   : {best_acc}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
colors  = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']

for ax, metric, color in zip(axes.flatten(), metrics, colors):
    sorted_df = comparison_df.sort_values(metric, ascending=True)
    bars = ax.barh(sorted_df['Model'], sorted_df[metric], color=color, edgecolor='black', alpha=0.85)
    ax.set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    ax.set_xlabel(f'{metric} (%)')
    ax.set_xlim(0, 110)
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.5, bar.get_y() + bar.get_height()/2,
                f'{width:.1f}%', va='center', ha='left', fontsize=9)

plt.suptitle('Model Performance Comparison — Bank Campaign Sense',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: model_comparison.png")

In [ ]:
best_model   = trained_models[best_model_name]
y_pred_best  = best_model.predict(X_test)
cm           = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No (0)', 'Yes (1)']).plot(
    cmap='Blues', ax=ax, colorbar=False
)
ax.set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 9: Feature Importance & Strategic Insights

In [ ]:
rf_model    = trained_models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X_encoded.columns)
top10       = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
colors_bar = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top10)))
bars = plt.barh(top10.index[::-1], top10.values[::-1], color=colors_bar, edgecolor='black')
plt.title('Top 10 Predictors for Customer Response\n(Random Forest Feature Importance)',
          fontsize=13, fontweight='bold')
plt.xlabel('Feature Importance Score')
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.002, bar.get_y() + bar.get_height()/2,
             f'{width:.4f}', va='center', ha='left', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: feature_importance.png")

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║          BANK CAMPAIGN SENSE — STRATEGIC INSIGHTS               ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  KEY FINDINGS:                                                   ║
║  1. 'duration' is the strongest predictor                        ║
║     → Longer calls = Higher subscription chance                  ║
║                                                                  ║
║  2. 'balance' is a key financial indicator                       ║
║     → Higher balance customers respond better                    ║
║                                                                  ║
║  3. 'age' plays a significant role                               ║
║     → Middle-aged and older customers convert more               ║
║                                                                  ║
║  4. 'poutcome_success' — prior positive outcome matters          ║
║     → Re-targeting past successes increases conversion           ║
║                                                                  ║
║  BEST MODEL: {}           ║
║  F1-Score: {}%  |  Accuracy: {}%           ║
║                                                                  ║
║  RECOMMENDATION:                                                 ║
║  Prioritize calling customers with:                              ║
║  → Higher account balances                                       ║
║  → Prior positive campaign interactions                          ║
║  → Longer average call durations in history                      ║
╚══════════════════════════════════════════════════════════════════╝""".format(
    best_model_name.ljust(20),
    str(best_f1).ljust(6),
    str(best_acc).ljust(6)
))